# importing packages

In [18]:
import pandas as pd
import os
import random
import numpy as np
import tensorflow as tf
import math
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Loading in data

In [19]:
from google.colab import files
uploaded = files.upload()

Saving Austria.xlsx to Austria (1).xlsx
Saving Belgium.xlsx to Belgium (1).xlsx
Saving Denmark.xlsx to Denmark (1).xlsx
Saving Finland.xlsx to Finland (1).xlsx
Saving France.xlsx to France (1).xlsx
Saving Germany.xlsx to Germany (1).xlsx
Saving Greece.xlsx to Greece (1).xlsx
Saving Ireland.xlsx to Ireland (1).xlsx
Saving Italy.xlsx to Italy (1).xlsx
Saving Luxembourg.xlsx to Luxembourg (1).xlsx
Saving Netherlands.xlsx to Netherlands (1).xlsx
Saving Portugal.xlsx to Portugal (1).xlsx
Saving Spain.xlsx to Spain (1).xlsx
Saving Sweden.xlsx to Sweden (1).xlsx


In [20]:
austria = pd.read_excel("Austria.xlsx")
belgium = pd.read_excel('Belgium.xlsx')
denmark = pd.read_excel('Denmark.xlsx')
finland = pd.read_excel('Finland.xlsx')
france = pd.read_excel('France.xlsx')
germany = pd.read_excel('Germany.xlsx')
ireland = pd.read_excel('Ireland.xlsx')
italy = pd.read_excel('Italy.xlsx')
luxembourg = pd.read_excel('Luxembourg.xlsx')
netherlands = pd.read_excel('Netherlands.xlsx')
portugal = pd.read_excel('Portugal.xlsx')
spain = pd.read_excel('Spain.xlsx')
sweden = pd.read_excel('Sweden.xlsx')

# Functions for running over multiple countries

Functions

In [21]:
def set_seeds(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ── low-level helpers ─────────────────────────────────────────────────────────
def _make_windows(series_2d: np.ndarray, lookback: int):
    X, y = [], []
    for i in range(lookback, len(series_2d)):
        X.append(series_2d[i - lookback:i, :])
        y.append(series_2d[i, :])
    return np.array(X), np.array(y)


def _build_model(lookback: int, units: int, dropout: float) -> tf.keras.Model:
    model = Sequential([
        Input(shape=(lookback, 1)),
        LSTM(units),
        Dropout(dropout),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model


def _recursive_forecast(model, seed_window: np.ndarray, h: int) -> np.ndarray:
    """
    Produce h-step-ahead forecasts by rolling the window forward using
    each prediction (no peeking at true values).

    Parameters
    ----------
    seed_window : array of shape (lookback, 1), in scaled space
    """
    window = seed_window.copy()
    lookback = window.shape[0]
    preds = []
    for _ in range(h):
        x_in = window.reshape(1, lookback, 1)
        yhat = model.predict(x_in, verbose=0)[0, 0]
        preds.append(yhat)
        window = np.vstack([window[1:, :], [[yhat]]])
    return np.array(preds)


def _fit_lstm(
    scaled_series: np.ndarray,
    lookback: int,
    units: int,
    dropout: float,
    epochs: int,
    batch_size: int,
    patience: int,
) -> tf.keras.Model:
    """Fit LSTM on a pre-scaled 2-D array (n, 1). No validation split."""
    X, y = _make_windows(scaled_series, lookback)
    model = _build_model(lookback, units, dropout)
    es = EarlyStopping(monitor="loss", patience=patience, restore_best_weights=True)
    model.fit(
        X, y,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[es],
        verbose=0,
        shuffle=False,
    )
    return model


# ── core PI function ──────────────────────────────────────────────────────────
def lstm_bootstrap_pi(
    df: pd.DataFrame,
    country: str,
    year_col: str = "Year",
    target_col: str = "gni",
    test_start_year: int = 2012,
    lookback: int = 5,
    units: int = 32,
    dropout: float = 0.2,
    epochs: int = 300,
    batch_size: int = 16,
    patience: int = 25,
    n_boot: int = 1000,
    alpha: float = 0.05,
    seed: int = 42,
) -> dict:
    """
    Build bootstrap prediction intervals for an LSTM on a single country.

    The procedure mirrors the R validation_errors() / nnetar bootstrap approach:
      1. Split train_df into core (80%) and validation (20%) by time.
      2. Fit LSTM on core only.
      3. Forecast the validation block recursively → compute residuals.
      4. Fit final LSTM on all of train_df.
      5. Forecast h = len(test_df) steps ahead; for each bootstrap replicate,
         add a resampled block of val residuals to the point forecasts.
      6. Compute PI bounds as empirical (alpha/2, 1-alpha/2) quantiles.

    Returns
    -------
    dict with keys:
        predictions : DataFrame  (year, true, pred, pi_lo, pi_hi)
        diagnostics : DataFrame  (country, coverage, rel_width_h1,
                                  rel_width_h5, rel_width_last)
        val_residuals : np.ndarray
    """
    set_seeds(seed)

    # ── 1. prepare data ───────────────────────────────────────────────────────
    d = df.copy()
    if year_col not in d.columns or target_col not in d.columns:
        raise ValueError(f"Missing columns: {year_col}, {target_col}")

    d = (d.sort_values(year_col)
          .reset_index(drop=True)
          .assign(**{year_col: lambda x: pd.to_numeric(x[year_col], errors="coerce"),
                     target_col: lambda x: pd.to_numeric(x[target_col], errors="coerce")})
          .dropna(subset=[year_col, target_col]))
    d[year_col] = d[year_col].astype(int)
    d = d.drop_duplicates(subset=[year_col], keep="last").reset_index(drop=True)

    train_df = d[d[year_col] < test_start_year].copy()
    test_df  = d[d[year_col] >= test_start_year].copy()

    if len(test_df) == 0:
        raise ValueError("Test set is empty.")
    if len(train_df) <= lookback:
        raise ValueError("Not enough training rows for chosen lookback.")

    # ── 2. core / validation split (mirrors R: n_val = ceil(0.2 * n)) ────────
    n_train = len(train_df)
    n_val   = math.ceil(0.2 * n_train)
    n_core  = n_train - n_val

    if n_core <= lookback:
        raise ValueError(
            f"Core set too small ({n_core} rows) for lookback={lookback}. "
            "Reduce lookback or use a longer training series."
        )

    core_values = train_df[target_col].to_numpy(dtype=float).reshape(-1, 1)[:n_core]
    val_values  = train_df[target_col].to_numpy(dtype=float)[n_core:]

    # ── 3. scale on core, fit on core ─────────────────────────────────────────
    scaler_core = MinMaxScaler()
    core_scaled = scaler_core.fit_transform(core_values)

    model_core = _fit_lstm(core_scaled, lookback, units, dropout,
                           epochs, batch_size, patience)

    # forecast validation block recursively (scaled space, then invert)
    seed_window_core = core_scaled[-lookback:, :]
    val_preds_scaled = _recursive_forecast(model_core, seed_window_core,
                                           h=len(val_values))
    val_preds = scaler_core.inverse_transform(
        val_preds_scaled.reshape(-1, 1)
    ).ravel()

    val_residuals = val_values - val_preds   # actual − forecast

    # ── 4. final model on full train_df ───────────────────────────────────────
    scaler_final = MinMaxScaler()
    train_scaled = scaler_final.fit_transform(
        train_df[target_col].to_numpy(dtype=float).reshape(-1, 1)
    )

    model_final = _fit_lstm(train_scaled, lookback, units, dropout,
                            epochs, batch_size, patience)

    # ── 5. point forecast on test set ────────────────────────────────────────
    h = len(test_df)
    seed_window_final = train_scaled[-lookback:, :]
    test_preds_scaled = _recursive_forecast(model_final, seed_window_final, h=h)
    y_pred = scaler_final.inverse_transform(
        test_preds_scaled.reshape(-1, 1)
    ).ravel()
    y_true = test_df[target_col].to_numpy(dtype=float)

    # ── 6. bootstrap PI ───────────────────────────────────────────────────────
    # For each replicate, draw h residuals with replacement from val_residuals
    # and add them to the point forecast (additive, same as R nnetar bootstrap).
    rng = np.random.default_rng(seed)
    boot_matrix = np.empty((n_boot, h))  # rows = replicates, cols = horizons

    for b in range(n_boot):
        sampled = rng.choice(val_residuals, size=h, replace=True)
        boot_matrix[b, :] = y_pred + sampled

    pi_lo = np.quantile(boot_matrix, alpha / 2,     axis=0)
    pi_hi = np.quantile(boot_matrix, 1 - alpha / 2, axis=0)

    # ── 7. assemble outputs ───────────────────────────────────────────────────
    pred_df = pd.DataFrame({
        year_col:               test_df[year_col].to_numpy(),
        f"{target_col}_true":   y_true,
        f"{target_col}_pred":   y_pred,
        "pi_lo":                pi_lo,
        "pi_hi":                pi_hi,
    })

    diag_df = _pi_diagnostics(country, y_true, y_pred, pi_lo, pi_hi)

    return {
        "predictions":  pred_df,
        "diagnostics":  diag_df,
        "val_residuals": val_residuals,
        "n_train":      n_train,
        "n_core":       n_core,
        "n_val":        n_val,
        "n_test":       h,
    }


# ── diagnostic helpers ────────────────────────────────────────────────────────
def _coverage_prob(y_test, pi_lo, pi_hi) -> float:
    return float(np.mean((y_test >= pi_lo) & (y_test <= pi_hi)))


def _relative_pi_width_h(y_test, pi_lo, pi_hi, h_idx: int):
    """Relative PI width at a specific 1-based horizon index."""
    i = h_idx - 1
    if i >= len(y_test):
        return np.nan
    denom = abs(y_test[i])
    if denom == 0:
        return np.nan
    return float((pi_hi[i] - pi_lo[i]) / denom)


def _relative_pi_width_last(y_test, pi_lo, pi_hi):
    return _relative_pi_width_h(y_test, pi_lo, pi_hi, len(y_test))


def _pi_diagnostics(
    country: str,
    y_test: np.ndarray,
    y_pred: np.ndarray,
    pi_lo: np.ndarray,
    pi_hi: np.ndarray,
) -> pd.DataFrame:
    return pd.DataFrame([{
        "country":           country,
        "n_test":            len(y_test),
        "coverage":          _coverage_prob(y_test, pi_lo, pi_hi),
        "rel_width_h1":      _relative_pi_width_h(y_test, pi_lo, pi_hi, 1),
        "rel_width_h5":      _relative_pi_width_h(y_test, pi_lo, pi_hi, 5),
        "rel_width_last":    _relative_pi_width_last(y_test, pi_lo, pi_hi),
    }])

# Running the code for all countries

In [48]:
SEED = 42
set_seeds(SEED)

result = lstm_bootstrap_pi(
    df=germany,
    country="germany",
    year_col="Year",
    target_col="unemployment",
    test_start_year=2014,
    lookback=1,
    units=4,
    dropout=0.1,
    epochs=300,
    batch_size=16,
    patience=25,
    n_boot=1000,
    alpha=0.05,
    seed=SEED,
)

In [49]:
result['diagnostics']

,country,n_test,coverage,rel_width_h1,rel_width_h5,rel_width_last
0,germany,11,0.181818,0.543172,0.801285,0.754035
